# Tutorial: Data Validation and Input Quality Control

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Data/model engineers maintaining exogenous datasets.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Run lint and schema validation checks.
- Inspect exogenous files for gaps/duplicates.
- Create quick QC summaries for key datasets.


## Outline

1. Run lint + schema validation scripts
2. Inventory exogenous files and columns
3. Check duplicate and missing year coverage
4. Run domain checks (bounds/rate sum patterns)
5. Save QC summary tables


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: Run built-in validation scripts


In [ ]:
_ = sh("python scripts/validation/lint_run_configs.py", cwd=REPO)
_ = sh(f"python scripts/validation/validate_exogenous_inputs.py --config {CONFIG}", cwd=REPO)


## Step 2: Inventory exogenous CSV files


In [ ]:
exo = REPO / "data" / "exogenous"
files = sorted(exo.glob("*.csv"))
print("file count:", len(files))
for p in files:
    print(" -", p.name)


## Step 3: Quick schema and null scan


In [ ]:
rows = []
for p in sorted((REPO / "data" / "exogenous").glob("*.csv")):
    df = pd.read_csv(p)
    rows.append({
        "file": p.name,
        "rows": len(df),
        "cols": len(df.columns),
        "null_cells": int(df.isna().sum().sum()),
        "columns": ", ".join(df.columns[:8]),
    })
scan = pd.DataFrame(rows).sort_values("file")
display(scan)


## Step 4: Duplicate-key checks for common key shapes


In [ ]:
def dup_count(df: pd.DataFrame, keys: list[str]) -> int:
    use = [k for k in keys if k in df.columns]
    if not use:
        return -1
    return int(df.duplicated(use).sum())

checks = []
for p in sorted((REPO / "data" / "exogenous").glob("*.csv")):
    df = pd.read_csv(p)
    checks.append({
        "file": p.name,
        "dup_year_material_region": dup_count(df, ["year", "material", "region"]),
        "dup_year_material_region_end_use": dup_count(df, ["year", "material", "region", "end_use"]),
    })
display(pd.DataFrame(checks).sort_values("file"))


## Step 5: Domain sanity checks for routing-like variables


In [ ]:
routing = REPO / "data" / "exogenous" / "collection_routing_rates.csv"
if routing.exists():
    df = pd.read_csv(routing)
    needed = ["recycling_rate", "remanufacturing_rate", "disposal_rate"]
    if all(c in df.columns for c in needed):
        df["routing_sum"] = df[needed].sum(axis=1)
        print(df["routing_sum"].describe())
        display(df[["year", "material", "region", "routing_sum"]].head())
else:
    print("No collection_routing_rates.csv found")


## Step 6: Persist QC summary artifact


In [ ]:
qc_dir = REPO / "outputs" / "analysis" / "data_qc"
qc_dir.mkdir(parents=True, exist_ok=True)
scan_path = qc_dir / "exogenous_file_scan.csv"
scan.to_csv(scan_path, index=False)
print("Wrote", scan_path)


## Pitfalls

- Passing schema validation does not guarantee realistic trajectories.
- Always inspect trend continuity and region/material consistency after schema checks.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
